In [0]:
CREATE CONNECTION IF NOT EXISTS databricks_earthquake_test TYPE HTTP OPTIONS (
  "host" = "https://earthquake.usgs.gov",
  "port" = "443",
  "base_path" = "/earthquakes/feed/v1.0/",
  "bearer_token" = "vi"
)

In [0]:
use catalog data_engineering;

use schema bronze;

create volume if not exists earthquake_data;

In [0]:
%python
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("databricks_earthquake_test")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
%python
dbutils.widgets.text('catalog_name','data_engineering','data_engineering')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
%python

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql("USE SCHEMA bronze")

spark.sql("CREATE VOLUME IF NOT EXISTS earthquake_data")

In [0]:
%python
import requests
import json
import datetime

url = f"{base_url.rstrip('/')}/summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)


"""
%python
import requests

url = f"{base_url.rstrip('/')}/summary/all_day.geojson"

print("URL:", url)

response = requests.get(url)
print("Status:", response.status_code)
print("Raw response:", response.text[:200])   # debug

data = response.json()
print(len(data['features']))

"""